# MortgageGuard: Mortgage Delinquency Early-Warning System

**Author:** Elvira Ongoue

## Business Problem

Mortgage-servicing teams have limited resources and cannot contact every borrower.
This project will develop a machine-learning system that identifies mortgages at
increased risk of becoming seriously delinquent.

## Primary Question

Can information available today predict whether a mortgage will become 90 or
more days delinquent within the following six months?

## Intended Use

The predictions will help prioritize borrowers for early support and outreach.
The model will not be used to approve or deny mortgages.

## Initial Success Criteria

The model should:

1. Identify a meaningful proportion of future serious delinquencies.
2. Perform better than a simple baseline.
3. Produce understandable risk explanations.
4. Avoid using information that would not have been available at prediction time.

## 1. Prototype the workflow on the sample file

The small sample is used to validate the delimiter, legacy field layout,
data types, delinquency definitions, and six-month target logic before the
workflow is scaled to the full quarterly file. Individual loan-level outputs
are intentionally omitted from this public notebook.


In [ ]:
import sys
import platform

print("Python version:", sys.version)
print("System:", platform.system())
print("MortgageGuard environment is ready.")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
System: Linux
MortgageGuard environment is ready.


In [ ]:
import pandas as pd

file_path = "/content/sf-loan-performance-data-sample.csv"

mortgage_sample = pd.read_csv(
    file_path,
    sep="|",
    header=None,
    dtype=str
)

print("Original shape:", mortgage_sample.shape)

Original shape: (757, 108)


In [ ]:
mortgage_sample = mortgage_sample.drop(
    columns=[mortgage_sample.columns[0], mortgage_sample.columns[-1]]
)

print("Cleaned shape:", mortgage_sample.shape)
print("Number of unique loans:", mortgage_sample.iloc[:, 0].nunique())

display(mortgage_sample.iloc[:5, :15])

In [ ]:
# Create temporary names matching this sample's 106-field layout
mortgage_sample.columns = [
    f"field_{position:03d}"
    for position in range(1, 107)
]

important_names = {
    "field_001": "loan_id",
    "field_002": "reporting_period",
    "field_003": "channel",
    "field_004": "seller_name",
    "field_005": "servicer_name",
    "field_006": "master_servicer",
    "field_007": "original_interest_rate",
    "field_008": "current_interest_rate",
    "field_009": "original_upb",
    "field_010": "upb_at_issuance",
    "field_011": "current_actual_upb",
    "field_012": "original_loan_term",
    "field_013": "origination_date",
    "field_014": "first_payment_date",
    "field_015": "loan_age",
    "field_016": "remaining_months_legal_maturity",
    "field_017": "remaining_months_maturity",
    "field_018": "maturity_date",
    "field_019": "original_ltv",
    "field_020": "original_cltv",
    "field_021": "number_of_borrowers",
    "field_022": "debt_to_income_ratio",
    "field_023": "borrower_credit_score",
    "field_024": "co_borrower_credit_score",
    "field_025": "first_time_homebuyer",
    "field_026": "loan_purpose",
    "field_027": "property_type",
    "field_028": "number_of_units",
    "field_029": "occupancy_status",
    "field_030": "property_state",
    "field_031": "msa",
    "field_032": "zip_code_short",
    "field_033": "mortgage_insurance_percentage",
    "field_034": "amortization_type",
    "field_039": "delinquency_status",
    "field_040": "loan_payment_history",
    "field_041": "modification_flag",
    "field_043": "zero_balance_code",
    "field_044": "zero_balance_date",
    "field_050": "last_paid_installment_date",
    "field_051": "foreclosure_date",
    "field_052": "disposition_date",
    "field_073": "servicing_activity_indicator",
    "field_078": "special_eligibility_program",
    "field_085": "property_valuation_method",
    "field_086": "high_balance_loan_indicator",
    "field_101": "borrower_assistance_plan",
    "field_102": "hltv_refinance_indicator",
    "field_105": "alternative_delinquency_resolution"
}

mortgage_sample = mortgage_sample.rename(columns=important_names)

In [ ]:
selected_columns = [
    "loan_id",
    "reporting_period",
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "loan_age",
    "borrower_credit_score",
    "property_state",
    "delinquency_status"
]

display(mortgage_sample[selected_columns].head(10))

In [ ]:
print("First reporting period:", mortgage_sample["reporting_period"].iloc[0])
print("Original interest rate:", mortgage_sample["original_interest_rate"].iloc[0])
print("Original loan balance:", mortgage_sample["original_upb"].iloc[0])
print("Original loan term:", mortgage_sample["original_loan_term"].iloc[0])


In [ ]:
mortgage_data = mortgage_sample.copy()

In [ ]:
date_columns = [
    "reporting_period",
    "origination_date",
    "first_payment_date",
    "maturity_date"
]

for column in date_columns:
    mortgage_data[column] = pd.to_datetime(
        mortgage_data[column],
        format="%m%Y",
        errors="coerce"
    )

In [ ]:
numeric_columns = [
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "original_loan_term",
    "loan_age",
    "remaining_months_legal_maturity",
    "remaining_months_maturity",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "debt_to_income_ratio",
    "borrower_credit_score",
    "co_borrower_credit_score",
    "number_of_units",
    "mortgage_insurance_percentage"
]

for column in numeric_columns:
    mortgage_data[column] = pd.to_numeric(
        mortgage_data[column],
        errors="coerce"
    )

In [ ]:
columns_to_check = [
    "loan_id",
    "reporting_period",
    "original_interest_rate",
    "original_upb",
    "original_loan_term",
    "loan_age"
]

display(mortgage_data[columns_to_check].head())

print("\nData types:")
print(mortgage_data[columns_to_check].dtypes)

In [ ]:
print("Number of rows:", mortgage_data.shape[0])
print("Number of columns:", mortgage_data.shape[1])
print("Number of loans:", mortgage_data["loan_id"].nunique())
print("Earliest reporting period:", mortgage_data["reporting_period"].min())
print("Latest reporting period:", mortgage_data["reporting_period"].max())

Number of rows: 757
Number of columns: 106
Number of loans: 8
Earliest reporting period: 2009-07-01 00:00:00
Latest reporting period: 2020-06-01 00:00:00


In [ ]:
duplicate_loan_months = mortgage_data.duplicated(
    subset=["loan_id", "reporting_period"]
).sum()

print("Duplicate loan-month records:", duplicate_loan_months)

Duplicate loan-month records: 0


In [ ]:
quality_columns = [
    "loan_id",
    "reporting_period",
    "original_interest_rate",
    "original_upb",
    "original_loan_term",
    "loan_age",
    "original_ltv",
    "debt_to_income_ratio",
    "borrower_credit_score",
    "property_state",
    "delinquency_status"
]

missing_summary = pd.DataFrame({
    "missing_count": mortgage_data[quality_columns].isna().sum(),
    "missing_percent": (
        mortgage_data[quality_columns].isna().mean() * 100
    ).round(2)
})

display(missing_summary)

,missing_count,missing_percent
loan_id,0,0.00
reporting_period,0,0.00
original_interest_rate,0,0.00
original_upb,0,0.00
original_loan_term,0,0.00
loan_age,5,0.66
original_ltv,0,0.00
debt_to_income_ratio,0,0.00
borrower_credit_score,0,0.00
property_state,0,0.00


In [ ]:
delinquency_counts = (
    mortgage_data["delinquency_status"]
    .value_counts(dropna=False)
    .sort_index()
)

display(delinquency_counts)

,count
delinquency_status,
00,652
01,18
02,9
03,7
04,4
05,3
06,3
07,3
08,3


In [ ]:
# Convert the original delinquency code into a number
mortgage_data["delinquency_status_numeric"] = pd.to_numeric(
    mortgage_data["delinquency_status"],
    errors="coerce"
)

# Begin with every value marked as unknown
mortgage_data["currently_seriously_delinquent"] = pd.Series(
    pd.NA,
    index=mortgage_data.index,
    dtype="Int64"
)

# Identify records whose delinquency status is known
known_status = mortgage_data["delinquency_status_numeric"].notna()

# Assign 1 for serious delinquency and 0 for non-serious delinquency
mortgage_data.loc[
    known_status,
    "currently_seriously_delinquent"
] = (
    mortgage_data.loc[
        known_status,
        "delinquency_status_numeric"
    ] >= 3
).astype(int)

In [ ]:
print(
    mortgage_data["currently_seriously_delinquent"]
    .value_counts(dropna=False)
)

currently_seriously_delinquent
0       679
1        74
<NA>      4
Name: count, dtype: Int64


In [ ]:
serious_by_loan = (
    mortgage_data
    .groupby("loan_id", as_index=False)
    .agg(
        monthly_records=("loan_id", "size"),
        serious_months=("currently_seriously_delinquent", "sum")
    )
)

serious_by_loan["ever_seriously_delinquent"] = (
    serious_by_loan["serious_months"] > 0
).astype(int)

display(serious_by_loan.drop(columns="loan_id").describe())


In [ ]:
first_serious_dates = (
    mortgage_data.loc[mortgage_data["currently_seriously_delinquent"] == 1]
    .groupby("loan_id", as_index=False)["reporting_period"]
    .min()
    .rename(columns={"reporting_period": "first_serious_delinquency_date"})
)

print("Loans with a serious-delinquency event:", len(first_serious_dates))
print("Earliest event:", first_serious_dates["first_serious_delinquency_date"].min())
print("Latest event:", first_serious_dates["first_serious_delinquency_date"].max())


In [ ]:
loan_to_inspect = first_serious_dates["loan_id"].iloc[0]

loan_history = (
    mortgage_data.loc[
        mortgage_data["loan_id"] == loan_to_inspect,
        [
            "loan_id",
            "reporting_period",
            "loan_age",
            "current_actual_upb",
            "delinquency_status",
            "delinquency_status_numeric",
            "currently_seriously_delinquent"
        ]
    ]
    .sort_values("reporting_period")
)

display(loan_history.tail(15))

In [ ]:
loan_to_inspect = first_serious_dates["loan_id"].iloc[0]

first_serious_date = first_serious_dates.loc[
    first_serious_dates["loan_id"] == loan_to_inspect,
    "first_serious_delinquency_date"
].iloc[0]

window_start = first_serious_date - pd.DateOffset(months=6)
window_end = first_serious_date + pd.DateOffset(months=3)

delinquency_window = (
    mortgage_data.loc[
        (mortgage_data["loan_id"] == loan_to_inspect)
        & (mortgage_data["reporting_period"] >= window_start)
        & (mortgage_data["reporting_period"] <= window_end),
        [
            "loan_id",
            "reporting_period",
            "loan_age",
            "current_actual_upb",
            "delinquency_status",
            "currently_seriously_delinquent"
        ]
    ]
    .sort_values("reporting_period")
)

display(delinquency_window)

In [ ]:
# Sort every mortgage chronologically
mortgage_data = mortgage_data.sort_values(
    ["loan_id", "reporting_period"]
).reset_index(drop=True)

# Create six columns representing the next six monthly outcomes
future_flags = pd.concat(
    [
        mortgage_data
        .groupby("loan_id")["currently_seriously_delinquent"]
        .shift(-month)
        for month in range(1, 7)
    ],
    axis=1
)

future_flags.columns = [
    "month_1",
    "month_2",
    "month_3",
    "month_4",
    "month_5",
    "month_6"
]

# Require six known future months
complete_six_month_horizon = (
    future_flags.notna().sum(axis=1) == 6
)

# Target equals 1 if any of the following six months is serious
mortgage_data["serious_delinquency_next_6_months"] = (
    future_flags.max(axis=1)
    .where(complete_six_month_horizon)
    .astype("Int64")
)

# We only predict for loans that are not already seriously delinquent
mortgage_data["eligible_for_prediction"] = (
    mortgage_data["delinquency_status_numeric"] < 3
)

In [ ]:
target_example = mortgage_data.loc[
    (mortgage_data["loan_id"] == loan_to_inspect)
    & (mortgage_data["reporting_period"] >= "2010-09-01")
    & (mortgage_data["reporting_period"] <= "2011-04-01"),
    [
        "reporting_period",
        "delinquency_status",
        "currently_seriously_delinquent",
        "serious_delinquency_next_6_months",
        "eligible_for_prediction"
    ]
]

display(target_example)

In [ ]:
modeling_sample = mortgage_data.loc[
    mortgage_data["eligible_for_prediction"]
    & mortgage_data["serious_delinquency_next_6_months"].notna()
].copy()

print("Eligible loan-month records:", len(modeling_sample))
print("Unique mortgages:", modeling_sample["loan_id"].nunique())

target_counts = (
    modeling_sample["serious_delinquency_next_6_months"]
    .value_counts()
    .sort_index()
)

target_percentages = (
    modeling_sample["serious_delinquency_next_6_months"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

display(target_summary)

Eligible loan-month records: 639
Unique mortgages: 8


,count,percentage
serious_delinquency_next_6_months,,
0,615,96.24
1,24,3.76


## 2. Load the full 2019 Q1 cohort

The full file is stored privately in Google Drive and is not distributed with
this repository. Update `private_folder` if your local Drive structure differs.
DuckDB is used to query the multi-gigabyte pipe-delimited file without loading
it completely into pandas memory.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

private_folder = Path("/content/drive/MyDrive/MortgageGuard/private_data")
zip_files = list(private_folder.glob("*.zip"))

print("ZIP files found:")

for file in zip_files:
    size_gb = file.stat().st_size / (1024**3)
    print(f"- {file.name}: {size_gb:.2f} GB")

ZIP files found:
- 2019Q1.csv.zip: 0.24 GB


In [ ]:
import zipfile

zip_path = zip_files[0]

with zipfile.ZipFile(zip_path, "r") as archive:
    members = archive.infolist()

    print("Files inside the ZIP:")
    for member in members:
        print(
            f"- {member.filename} | "
            f"uncompressed size: {member.file_size / (1024**3):.2f} GB"
        )

    csv_member = members[0]

    with archive.open(csv_member) as file:
        first_line = file.readline().decode("utf-8").strip()

print("\nFirst 300 characters:")
print(first_line[:300])

print("\nNumber of pipe-separated fields:")
print(len(first_line.split("|")))

In [ ]:
import zipfile

zip_path = private_folder / "2019Q1.csv.zip"

with zipfile.ZipFile(zip_path, "r") as archive:
    # Select the real CSV and ignore the hidden Mac metadata file
    csv_member = next(
        name for name in archive.namelist()
        if name.endswith(".csv") and not name.startswith("__MACOSX/")
    )

    with archive.open(csv_member) as file:
        for row_number in range(1, 6):
            line = file.readline().decode("utf-8").rstrip("\r\n")
            fields = line.split("|")

            begins_with_pipe = fields[0] == ""
            ends_with_pipe = fields[-1] == ""

            # Remove only the empty boundary fields
            real_fields = fields[
                1 if begins_with_pipe else 0:
                -1 if ends_with_pipe else None
            ]

            print(
                f"Row {row_number}: "
                f"{len(fields)} sections, "
                f"{len(real_fields)} actual fields"
            )

print("\nSelected CSV:", csv_member)
print("Begins with pipe:", begins_with_pipe)
print("Ends with pipe:", ends_with_pipe)

Row 1: 113 sections, 111 actual fields
Row 2: 113 sections, 111 actual fields
Row 3: 113 sections, 111 actual fields
Row 4: 113 sections, 111 actual fields
Row 5: 113 sections, 111 actual fields

Selected CSV: 2019Q1.csv
Begins with pipe: True
Ends with pipe: True


In [ ]:
import shutil

total, used, free = shutil.disk_usage("/content")

print(f"Total Colab storage: {total / (1024**3):.2f} GB")
print(f"Used storage: {used / (1024**3):.2f} GB")
print(f"Free storage: {free / (1024**3):.2f} GB")

required_gb = 5

if free / (1024**3) >= required_gb:
    print("\nEnough temporary storage. Ready for extraction.")
else:
    print("\nNot enough temporary storage. Do not extract yet.")

Total Colab storage: 225.83 GB
Used storage: 20.35 GB
Free storage: 205.47 GB

Enough temporary storage. Ready for extraction.


In [ ]:
import zipfile
from pathlib import Path

zip_path = private_folder / "2019Q1.csv.zip"
local_csv_path = Path("/content/2019Q1.csv")

with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open("2019Q1.csv") as source:
        with open(local_csv_path, "wb") as destination:
            shutil.copyfileobj(source, destination)

print("Extraction complete.")
print("Location:", local_csv_path)
print(f"CSV size: {local_csv_path.stat().st_size / (1024**3):.2f} GB")

Extraction complete.
Location: /content/2019Q1.csv
CSV size: 3.27 GB


In [ ]:
 !pip -q install duckdb

import duckdb

connection = duckdb.connect("/content/mortgageguard.duckdb")

print("DuckDB version:", duckdb.__version__)
print("Database connection ready.")

DuckDB version: 1.3.2
Database connection ready.


In [ ]:
csv_path = "/content/2019Q1.csv"

column_structure = connection.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_csv(
        '{csv_path}',
        delim = '|',
        header = false,
        all_varchar = true,
        null_padding = true
    )
""").df()

print("Columns detected:", len(column_structure))
display(column_structure)

Columns detected: 113


,column_name,column_type,null,key,default,extra
0,column000,VARCHAR,YES,None,None,None
1,column001,VARCHAR,YES,None,None,None
2,column002,VARCHAR,YES,None,None,None
3,column003,VARCHAR,YES,None,None,None
4,column004,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
108,column108,VARCHAR,YES,None,None,None
109,column109,VARCHAR,YES,None,None,None
110,column110,VARCHAR,YES,None,None,None
111,column111,VARCHAR,YES,None,None,None


In [ ]:
connection.execute(f"""
    CREATE OR REPLACE VIEW mortgage_raw AS
    SELECT
        column001 AS loan_id,
        TRY_STRPTIME(column002, '%m%Y') AS reporting_period,
        column003 AS channel,
        column004 AS seller_name,
        column005 AS servicer_name,
        TRY_CAST(column007 AS DOUBLE) AS original_interest_rate,
        TRY_CAST(column008 AS DOUBLE) AS current_interest_rate,
        TRY_CAST(column009 AS DOUBLE) AS original_upb,
        TRY_CAST(column011 AS DOUBLE) AS current_actual_upb,
        TRY_CAST(column012 AS INTEGER) AS original_loan_term,
        TRY_STRPTIME(column013, '%m%Y') AS origination_date,
        TRY_STRPTIME(column014, '%m%Y') AS first_payment_date,
        TRY_CAST(column015 AS INTEGER) AS loan_age,
        TRY_CAST(column019 AS DOUBLE) AS original_ltv,
        TRY_CAST(column020 AS DOUBLE) AS original_cltv,
        TRY_CAST(column022 AS DOUBLE) AS debt_to_income_ratio,
        TRY_CAST(column023 AS INTEGER) AS borrower_credit_score,
        column026 AS loan_purpose,
        column027 AS property_type,
        TRY_CAST(column028 AS INTEGER) AS number_of_units,
        column029 AS occupancy_status,
        column030 AS property_state,
        column039 AS delinquency_status
    FROM read_csv(
        '{csv_path}',
        delim = '|',
        header = false,
        all_varchar = true,
        null_padding = true
    )
""")

print("Mortgage view created.")

Mortgage view created.


In [ ]:
preview = connection.execute("""
    SELECT * EXCLUDE (loan_id)
    FROM mortgage_raw
    LIMIT 5
""").df()

display(preview)
print("Preview shape:", preview.shape)

In [ ]:
parquet_path = "/content/mortgage_2019q1_selected.parquet"

connection.execute(f"""
    COPY (
        SELECT *
        FROM mortgage_raw
    )
    TO '{parquet_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

from pathlib import Path

parquet_size_gb = Path(parquet_path).stat().st_size / (1024**3)

print("Parquet file created.")
print("Location:", parquet_path)
print(f"Compressed size: {parquet_size_gb:.2f} GB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet file created.
Location: /content/mortgage_2019q1_selected.parquet
Compressed size: 0.05 GB


## 3. Validate the full dataset

This section verifies record counts, date coverage, uniqueness of
loan-month keys, delinquency codes, and loan-level outcomes.


In [ ]:
connection.execute(f"""
    CREATE OR REPLACE VIEW mortgage_data AS
    SELECT *
    FROM read_parquet('{parquet_path}')
""")

dataset_summary = connection.execute("""
    SELECT
        COUNT(*) AS monthly_records,
        COUNT(DISTINCT loan_id) AS unique_mortgages,
        MIN(reporting_period) AS earliest_reporting_period,
        MAX(reporting_period) AS latest_reporting_period,
        COUNT(*) - COUNT(loan_id) AS missing_loan_ids,
        COUNT(*) - COUNT(reporting_period) AS missing_reporting_periods
    FROM mortgage_data
""").df()

display(dataset_summary)

,monthly_records,unique_mortgages,earliest_reporting_period,latest_reporting_period,missing_loan_ids,missing_reporting_periods
0,11774637,341865,2019-01-01,2026-03-01,0,0


In [ ]:
duplicate_summary = connection.execute("""
    SELECT
        COUNT(*) AS duplicate_loan_month_groups,
        COALESCE(SUM(record_count - 1), 0) AS extra_duplicate_records
    FROM (
        SELECT
            loan_id,
            reporting_period,
            COUNT(*) AS record_count
        FROM mortgage_data
        GROUP BY loan_id, reporting_period
        HAVING COUNT(*) > 1
    )
""").df()

display(duplicate_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_loan_month_groups,extra_duplicate_records
0,0,0.0


In [ ]:
delinquency_distribution = connection.execute("""
    SELECT
        COALESCE(delinquency_status, 'MISSING') AS delinquency_status,
        COUNT(*) AS monthly_records,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            4
        ) AS percentage
    FROM mortgage_data
    GROUP BY delinquency_status
    ORDER BY
        TRY_CAST(delinquency_status AS INTEGER) NULLS LAST,
        delinquency_status
""").df()

display(delinquency_distribution)

,delinquency_status,monthly_records,percentage
0,00,11226978,95.3488
1,01,162761,1.3823
2,02,60749,0.5159
3,03,38672,0.3284
4,04,28365,0.2409
...,...,...,...
79,79,2,0.0000
80,80,2,0.0000
81,81,2,0.0000
82,82,1,0.0000


In [ ]:
connection.execute("""
    CREATE OR REPLACE VIEW mortgage_clean AS
    SELECT
        *,
        TRY_CAST(delinquency_status AS INTEGER)
            AS delinquency_status_numeric,

        CASE
            WHEN TRY_CAST(delinquency_status AS INTEGER) IS NULL
                THEN NULL
            WHEN TRY_CAST(delinquency_status AS INTEGER) >= 3
                THEN 1
            ELSE 0
        END AS currently_seriously_delinquent

    FROM mortgage_data
""")

print("Clean mortgage view created.")

Clean mortgage view created.


In [ ]:
serious_summary = connection.execute("""
    SELECT
        CASE
            WHEN currently_seriously_delinquent = 0
                THEN 'Not seriously delinquent'
            WHEN currently_seriously_delinquent = 1
                THEN 'Seriously delinquent'
            ELSE 'Unknown status'
        END AS category,
        COUNT(*) AS monthly_records,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            3
        ) AS percentage
    FROM mortgage_clean
    GROUP BY currently_seriously_delinquent
    ORDER BY currently_seriously_delinquent NULLS LAST
""").df()

display(serious_summary)

,category,monthly_records,percentage
0,Not seriously delinquent,11450488,97.247
1,Seriously delinquent,237840,2.020
2,Unknown status,86309,0.733


In [ ]:
loan_outcome_summary = connection.execute("""
    WITH loan_outcomes AS (
        SELECT
            loan_id,
            MAX(currently_seriously_delinquent)
                AS ever_seriously_delinquent
        FROM mortgage_clean
        GROUP BY loan_id
    )

    SELECT
        CASE
            WHEN ever_seriously_delinquent = 1
                THEN 'Ever seriously delinquent'
            WHEN ever_seriously_delinquent = 0
                THEN 'Never seriously delinquent'
            ELSE 'No known delinquency status'
        END AS loan_outcome,
        COUNT(*) AS mortgages,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            3
        ) AS percentage
    FROM loan_outcomes
    GROUP BY ever_seriously_delinquent
    ORDER BY ever_seriously_delinquent NULLS LAST
""").df()

display(loan_outcome_summary)

,loan_outcome,mortgages,percentage
0,Never seriously delinquent,318980,93.306
1,Ever seriously delinquent,22885,6.694


In [ ]:
connection.execute("""
    CREATE OR REPLACE TABLE serious_delinquency_events AS
    SELECT
        loan_id,
        MIN(reporting_period) AS first_serious_delinquency_date,
        COUNT(*) AS serious_months
    FROM mortgage_clean
    WHERE currently_seriously_delinquent = 1
    GROUP BY loan_id
""")

event_summary = connection.execute("""
    SELECT
        COUNT(*) AS mortgages_with_event,
        MIN(first_serious_delinquency_date) AS earliest_event,
        MAX(first_serious_delinquency_date) AS latest_event
    FROM serious_delinquency_events
""").df()

display(event_summary)


In [ ]:
first_serious_by_year = connection.execute("""
    SELECT
        EXTRACT(YEAR FROM first_serious_delinquency_date)::INTEGER
            AS first_serious_year,
        COUNT(*) AS mortgages
    FROM serious_delinquency_events
    GROUP BY first_serious_year
    ORDER BY first_serious_year
""").df()

display(first_serious_by_year)

,first_serious_year,mortgages
0,2019,872
1,2020,16771
2,2021,2598
3,2022,937
4,2023,612
5,2024,535
6,2025,433
7,2026,127


## 4. Construct the six-month prediction target

Eligible observations must be below serious delinquency at the prediction
date and have six consecutive future months with known outcomes. Records
after a loan's first serious-delinquency event are excluded.


In [ ]:
connection.execute("""
    CREATE OR REPLACE TABLE mortgage_labeled AS

    WITH future_windows AS (
        SELECT
            *,

            LEAD(reporting_period, 6) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
            ) AS sixth_future_period,

            COUNT(currently_seriously_delinquent) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
                ROWS BETWEEN 1 FOLLOWING AND 6 FOLLOWING
            ) AS known_future_months,

            MAX(currently_seriously_delinquent) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
                ROWS BETWEEN 1 FOLLOWING AND 6 FOLLOWING
            ) AS future_serious_max

        FROM mortgage_clean
    ),

    target_created AS (
        SELECT
            future_windows.*,

            CASE
                WHEN sixth_future_period =
                     reporting_period + INTERVAL '6 months'
                 AND known_future_months = 6
                THEN future_serious_max
                ELSE NULL
            END AS serious_delinquency_next_6_months

        FROM future_windows
    )

    SELECT
        target_created.*,

        CASE
            WHEN delinquency_status_numeric BETWEEN 0 AND 2
             AND serious_delinquency_next_6_months IS NOT NULL
             AND (
                    first_serious_delinquency_date IS NULL
                    OR reporting_period < first_serious_delinquency_date
                 )
            THEN TRUE
            ELSE FALSE
        END AS eligible_for_prediction

    FROM target_created
    LEFT JOIN serious_delinquency_events
        USING (loan_id)
""")

print("Six-month prediction target created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Six-month prediction target created.


In [ ]:
modeling_summary = connection.execute("""
    SELECT
        COUNT(*) AS eligible_loan_months,
        COUNT(DISTINCT loan_id) AS eligible_mortgages,

        SUM(
            CASE
                WHEN serious_delinquency_next_6_months = 1 THEN 1
                ELSE 0
            END
        ) AS positive_records,

        SUM(
            CASE
                WHEN serious_delinquency_next_6_months = 0 THEN 1
                ELSE 0
            END
        ) AS negative_records,

        ROUND(
            100.0 * AVG(serious_delinquency_next_6_months),
            3
        ) AS positive_percentage

    FROM mortgage_labeled
    WHERE eligible_for_prediction = TRUE
""").df()

display(modeling_summary)

,eligible_loan_months,eligible_mortgages,positive_records,negative_records,positive_percentage
0,8918849,319125,132695.0,8786154.0,1.488


In [ ]:
target_by_year = connection.execute("""
    SELECT
        EXTRACT(YEAR FROM reporting_period)::INTEGER
            AS observation_year,
        COUNT(*) AS eligible_records,
        SUM(serious_delinquency_next_6_months)
            AS positive_records,
        ROUND(
            100.0 * AVG(serious_delinquency_next_6_months),
            3
        ) AS positive_percentage
    FROM mortgage_labeled
    WHERE eligible_for_prediction = TRUE
    GROUP BY observation_year
    ORDER BY observation_year
""").df()

display(target_by_year)

,observation_year,eligible_records,positive_records,positive_percentage
0,2019,3028182,16015.0,0.529
1,2020,1976297,94138.0,4.763
2,2021,1089329,9645.0,0.885
3,2022,850967,4744.0,0.557
4,2023,772350,3300.0,0.427
5,2024,708671,2922.0,0.412
6,2025,493053,1931.0,0.392


In [ ]:
from pathlib import Path

processed_folder = Path(
    "/content/drive/MyDrive/MortgageGuard/private_data/processed"
)
processed_folder.mkdir(parents=True, exist_ok=True)

modeling_base_path = (
    processed_folder / "2019Q1_modeling_base.parquet"
)

connection.execute(f"""
    COPY (
        SELECT *
        FROM mortgage_labeled
        WHERE eligible_for_prediction = TRUE
    )
    TO '{modeling_base_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

size_mb = modeling_base_path.stat().st_size / (1024**2)

print("Private modeling checkpoint saved.")
print("Location:", modeling_base_path)
print(f"Size: {size_mb:.2f} MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Private modeling checkpoint saved.
Location: /content/drive/MyDrive/MortgageGuard/private_data/processed/2019Q1_modeling_base.parquet
Size: 48.20 MB


## 5. Audit candidate predictors and define active loans

Candidate features are reviewed for missingness and implausible values.
Zero-balance observations are removed because they are not actionable for
active mortgage-servicing outreach.


In [ ]:
feature_audit = connection.execute("""
    SUMMARIZE
    SELECT
        original_interest_rate,
        current_interest_rate,
        original_upb,
        current_actual_upb,
        original_loan_term,
        loan_age,
        original_ltv,
        original_cltv,
        debt_to_income_ratio,
        borrower_credit_score,
        number_of_units,
        delinquency_status_numeric
    FROM mortgage_labeled
    WHERE eligible_for_prediction = TRUE
""").df()

display(feature_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,original_interest_rate,DOUBLE,2.75,6.875,1147,4.9549429852315585,0.5223966871079098,4.625,4.875,5.25,8918849,0.00
1,current_interest_rate,DOUBLE,2.75,6.875,1147,4.954921794156342,0.5224131491931462,4.625,4.875,5.25,8918849,0.00
2,original_upb,DOUBLE,14000.0,1387000.0,880,199187.0077629972,112735.35015115449,116295.1076851184,175224.79480300262,256220.02155481974,8918849,0.00
3,current_actual_upb,DOUBLE,0.0,1374592.6,4373428,141786.59051531085,118642.65330134002,54470.174337352844,125805.94605290322,206185.2702094247,8918849,0.00
4,original_loan_term,INTEGER,72,360,79,330.60231325813453,65.83405589183734,360,360,360,8918849,0.00
5,loan_age,INTEGER,-1,103,102,26.417761305298477,22.80778125414409,7,19,43,8918849,0.00
6,original_ltv,DOUBLE,2.0,97.0,131,76.1904598900598,18.618338680634405,67.30822079111695,80.0,93.4716397099954,8918849,0.00
7,original_cltv,DOUBLE,2.0,117.0,155,76.7052071405178,18.911559883413442,67.65579973719844,80.0,94.67817137553799,8918849,0.00
8,debt_to_income_ratio,DOUBLE,0.0,50.0,54,36.920277131161825,9.433313757783385,30.405317751012884,38.781194607977746,44.887738595680226,8918849,0.02
9,borrower_credit_score,INTEGER,620,834,201,741.8824315708317,48.50233802099129,706,748,783,8918849,0.15


In [ ]:
zero_balance_audit = connection.execute("""
    SELECT
        CASE
            WHEN current_actual_upb = 0
                THEN 'Zero current balance'
            WHEN current_actual_upb > 0
                THEN 'Positive current balance'
            ELSE 'Missing or invalid balance'
        END AS balance_category,

        COUNT(*) AS eligible_records,
        COUNT(DISTINCT loan_id) AS mortgages,
        SUM(serious_delinquency_next_6_months)
            AS positive_records,

        ROUND(
            100.0 * AVG(serious_delinquency_next_6_months),
            4
        ) AS positive_percentage

    FROM mortgage_labeled
    WHERE eligible_for_prediction = TRUE
    GROUP BY balance_category
    ORDER BY balance_category
""").df()

display(zero_balance_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,balance_category,eligible_records,mortgages,positive_records,positive_percentage
0,Positive current balance,7145733,265370,129486.0,1.8121
1,Zero current balance,1773116,319125,3209.0,0.1810


In [ ]:
connection.execute("""
    CREATE OR REPLACE VIEW modeling_population AS
    SELECT *
    FROM mortgage_labeled
    WHERE eligible_for_prediction = TRUE
      AND current_actual_upb > 0
""")

active_summary = connection.execute("""
    SELECT
        COUNT(*) AS active_loan_months,
        COUNT(DISTINCT loan_id) AS unique_mortgages,
        SUM(serious_delinquency_next_6_months)
            AS positive_records,
        COUNT(*) - SUM(serious_delinquency_next_6_months)
            AS negative_records,
        ROUND(
            100.0 * AVG(serious_delinquency_next_6_months),
            3
        ) AS positive_percentage
    FROM modeling_population
""").df()

display(active_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,active_loan_months,unique_mortgages,positive_records,negative_records,positive_percentage
0,7145733,265370,129486.0,7016247.0,1.812


### Active-loan eligibility decision

The initial eligible population included records with a current unpaid
principal balance of zero. These observations are generally not actionable
for a mortgage-servicing early-warning system because there is no active
balance requiring outreach.

Zero-balance observations also had a substantially lower six-month serious
delinquency rate than positive-balance observations (0.181% versus 1.812%).
Therefore, the final modeling population was restricted to records with a
current actual UPB greater than zero. This retained approximately 97.6% of
the positive target observations while aligning the dataset with the
project's business use case.

## 6. Engineer leakage-safe model features

All model inputs are available on or before the prediction month. Recent
delinquency history is calculated within each mortgage's chronological
record, and continuity is verified before export.


In [ ]:
connection.execute("""
    CREATE OR REPLACE TABLE model_features AS

    WITH history_features AS (
        SELECT
            *,

            LAG(delinquency_status_numeric, 1) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
            ) AS delinquency_status_1_month_ago,

            LAG(delinquency_status_numeric, 2) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
            ) AS delinquency_status_2_months_ago,

            SUM(
                CASE
                    WHEN delinquency_status_numeric > 0 THEN 1
                    ELSE 0
                END
            ) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS delinquent_months_last_3

        FROM mortgage_labeled
    )

    SELECT
        -- Identifiers used for splitting, not model inputs
        loan_id,
        reporting_period,

        -- Current loan condition
        current_interest_rate,
        current_actual_upb,
        loan_age,
        delinquency_status_numeric,

        -- Original borrower and loan characteristics
        original_interest_rate,
        original_upb,
        original_loan_term,
        original_ltv,
        original_cltv,
        debt_to_income_ratio,
        borrower_credit_score,
        number_of_units,

        -- Categorical characteristics
        channel,
        loan_purpose,
        property_type,
        occupancy_status,
        property_state,

        -- Engineered financial features
        current_interest_rate - original_interest_rate
            AS interest_rate_change,

        current_actual_upb / NULLIF(original_upb, 0)
            AS current_to_original_balance_ratio,

        1 - (
            current_actual_upb / NULLIF(original_upb, 0)
        ) AS principal_reduction_ratio,

        -- Recent payment-behavior features
        delinquency_status_1_month_ago,
        delinquency_status_2_months_ago,
        delinquent_months_last_3,

        -- Prediction target
        serious_delinquency_next_6_months

    FROM history_features

    WHERE eligible_for_prediction = TRUE
      AND current_actual_upb > 0
""")

print("Leakage-safe feature table created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leakage-safe feature table created.


In [ ]:
feature_preview = connection.execute("""
    SELECT * EXCLUDE (loan_id)
    FROM model_features
    LIMIT 10
""").df()

display(feature_preview)

row_count = connection.execute(
    "SELECT COUNT(*) FROM model_features"
).fetchone()[0]

column_count = connection.execute(
    "SELECT COUNT(*) FROM (DESCRIBE model_features)"
).fetchone()[0]

print("Shape:", (row_count, column_count))


In [ ]:
history_gap_audit = connection.execute("""
    WITH previous_dates AS (
        SELECT
            loan_id,
            reporting_period,

            LAG(reporting_period, 1) OVER (
                PARTITION BY loan_id
                ORDER BY reporting_period
            ) AS previous_reporting_period

        FROM mortgage_labeled
    )

    SELECT
        COUNT(*) AS active_records_with_previous_history,

        SUM(
            CASE
                WHEN previous_reporting_period =
                     reporting_period - INTERVAL '1 month'
                THEN 1
                ELSE 0
            END
        ) AS exact_one_month_history,

        SUM(
            CASE
                WHEN previous_reporting_period !=
                     reporting_period - INTERVAL '1 month'
                THEN 1
                ELSE 0
            END
        ) AS history_gaps,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN previous_reporting_period !=
                         reporting_period - INTERVAL '1 month'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            4
        ) AS gap_percentage

    FROM previous_dates
    JOIN modeling_population
        USING (loan_id, reporting_period)

    WHERE previous_reporting_period IS NOT NULL
""").df()

display(history_gap_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,active_records_with_previous_history,exact_one_month_history,history_gaps,gap_percentage
0,7145733,7145733.0,0.0,0.0


In [ ]:
feature_path = (
    processed_folder / "2019Q1_model_features.parquet"
)

connection.execute(f"""
    COPY model_features
    TO '{feature_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

feature_size_mb = feature_path.stat().st_size / (1024**2)

print("Feature checkpoint saved.")
print("Location:", feature_path)
print(f"Size: {feature_size_mb:.2f} MB")
print("Final shape: (7,145,733, 26)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature checkpoint saved.
Location: /content/drive/MyDrive/MortgageGuard/private_data/processed/2019Q1_model_features.parquet
Size: 135.77 MB
Final shape: (7,145,733, 26)


## Preparation outcome

The final feature table contains **7,145,733 active loan-month observations**
across **265,370 mortgages** and 26 columns. It is saved privately as Parquet
for the exploratory analysis and modeling notebook. Raw and processed data
are excluded from the public repository.
